In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from geobr import read_municipality
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('dados', exist_ok=True)

print("Baixando dados dos municípios de MG...")
mg = read_municipality(code_muni="MG", year=2020)
mg = mg.to_crs(epsg=31983)
mg['area_km2'] = mg.geometry.area / (10**6)
mg.to_file('dados/municipios-mg.geojson', driver='GeoJSON')
print("✅ 'municipios-mg.geojson' salvo!")

print("Gerando dados socioeconômicos simulados...")
nomes_muni = mg['name_muni'].unique()
df_ibge = pd.DataFrame({
    'municipio': nomes_muni,
    'populacao_2022': np.random.randint(5000, 2500000, len(nomes_muni)),
    'pib_mil_reais': np.random.randint(10000, 90000000, len(nomes_muni))
})
df_ibge.to_csv('dados/populacao-pib-municipios-mg.csv', index=False)
print("✅ 'populacao-pib-municipios-mg.csv' salvo!")

print("Unificando focos de desmatamento...")
# Aqui ele pega os arquivos que você gerou na Célula 2
ago = gpd.read_file('dados/desmatamento_ago22.gpkg')
setem = gpd.read_file('dados/desmatamento_set_22.gpkg')
ago['mes'] = 'Agosto'
setem['mes'] = 'Setembro'

focos = pd.concat([ago, setem], ignore_index=True)
focos = gpd.GeoDataFrame(focos, geometry='geometry', crs="EPSG:4326")
focos = focos.to_crs(epsg=31983)
focos.to_file('dados/focos-desmatamento-mg.geojson', driver='GeoJSON')
print("✅ 'focos-desmatamento-mg.geojson' salvo com sucesso!")